# Session 4: Ridge and the Tree Benchmark
**Emission-trajectory ML project. Dr. Khawar Naeem, QTTSC, Qatar University.**

First actual machine learning. Two model families enter, each in two parameterizations (predict the level, predict the change), all scored by `src/evaluate.py` on the same rows as the Session 3 baselines.

**Session contract:** models are fitted on TRAIN (targets through 2014), selected on VALIDATION (2015-2018), and the test years remain untouched. Deliverables: this notebook executed, an updated `results/model_comparison.csv` with six rows, and your check-question answers.

**Prerequisite:** Session 3 executed, so you know the baseline numbers these models must beat.

## 0. Pre-registration (fill in BEFORE running the models)

1. Ridge predicting the LEVEL: does it beat persistence on validation MAE? By a lot or a hair? Why? (Hint: what will Ridge learn the coefficient on `co2` to be?)
   > *your answer*
2. Ridge predicting the DELTA: better or worse than Ridge-on-level? Why might forcing the model to explain the CHANGE help or hurt?
   > *your answer*
3. Does the gradient-boosted tree beat Ridge? On MAE, on RMSE, or both?
   > *your answer*
4. Rank all six (2 baselines, 4 models) by validation MAE, best to worst. Commit.
   > *your answer*

## 1. Concepts before code

**Why Ridge and not plain OLS.** Our features are heavily correlated by construction: co2, its four lags, two rolling means, and the slope all move together. OLS with correlated regressors produces huge, unstable, opposite-signed coefficients (it can add 1000×lag1 and subtract 998×lag3). Ridge adds a penalty on coefficient size (alpha × sum of squared coefficients), which trades a little bias for a lot of stability. Alpha is chosen on validation, never on test.

**Why scaling matters for Ridge and not for trees.** Ridge's penalty is on coefficient SIZE, and coefficient size depends on feature units: a feature measured in millions needs a tiny coefficient, one in percentages needs a large one, so an unscaled penalty punishes them unequally. Standardizing puts all features on comparable scales so the penalty is fair. Trees never take a linear combination; they ask "is feature X < threshold?", and any monotonic rescaling leaves every split decision identical.

**Bagging versus boosting, one sentence each.** Bagging (random forest) trains many deep trees independently on bootstrap samples and averages them, reducing variance. Boosting trains shallow trees SEQUENTIALLY, each one fitted to the errors the ensemble still makes, reducing bias step by step. We use `HistGradientBoostingRegressor`: histogram binning makes split-finding fast, and it handles missing values natively, which is exactly what our nullable energy columns need.

**The leakage rule for preprocessing.** The imputer and scaler are fitted on TRAINING rows only, then applied unchanged to validation and test. If the imputer saw 2015-2018 medians, information from the validation years would flow into the model through the preprocessing, an invisible leak that inflates results. The sklearn `Pipeline` makes this automatic: `fit` on train fits every stage; `predict` on val only transforms.

## 2. Load data, define features and splits

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import evaluate

SEED = 42

df = pd.read_csv("../data/processed/modeling_table.csv")

# Features: everything the model may see at year t. Deliberately excluded:
# 'year' itself (a tree would memorize era boundaries it cannot extrapolate),
# identifiers, targets, and split labels.
CORE = ["co2", "co2_lag1", "co2_lag3", "co2_lag5", "co2_lag10",
        "co2_roll5_mean", "co2_roll10_mean", "co2_slope5",
        "co2_per_capita", "share_global_co2",
        "population", "pop_growth5_pct", "cement_co2", "flaring_co2"]
NULLABLE = ["primary_energy_consumption", "energy_growth1_pct", "energy_per_capita"]
FEATURES = CORE + NULLABLE

train = df[df["split"] == "train"].copy()
val   = df[df["split"] == "val"].copy()
print(f"train {len(train)} rows (targets <= 2014), val {len(val)} rows (targets 2015-2018), {len(FEATURES)} features")

## 3. Ridge, level parameterization

The pipeline: median imputer (for the nullable energy columns) -> standard scaler -> Ridge. All three stages fit on train only. Alpha is picked by validation MAE from a small, honest grid; we record the winner rather than searching hundreds of values.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

def fit_ridge(target_col, alphas=(0.1, 1.0, 10.0, 100.0)):
    """Fit Ridge on train for each alpha, return {alpha: val predictions of the LEVEL}.

    If target_col is 'target_delta', predictions are converted back to the
    level (co2 + predicted change) BEFORE scoring, so every model in this
    project is compared on the same quantity.
    """
    out = {}
    for a in alphas:
        pipe = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", Ridge(alpha=a, random_state=SEED)),
        ])
        pipe.fit(train[FEATURES], train[target_col])
        pred = pipe.predict(val[FEATURES])
        out[a] = pred if target_col == "target_co2_next" else val["co2"].to_numpy() + pred
    return out

ridge_level_by_alpha = fit_ridge("target_co2_next")
for a, p in ridge_level_by_alpha.items():
    print(f"alpha={a:>6}: val MAE {evaluate.mae(val['target_co2_next'], p):.3f}")

In [ ]:
# Freeze the chosen alpha by the printout above, then store the predictions.
RIDGE_LEVEL_ALPHA = ...   # <- fill in after reading the alphas printout
val["pred_ridge_level"] = ridge_level_by_alpha[RIDGE_LEVEL_ALPHA]

## 4. Ridge, delta parameterization (Amendment 2 in action)

Same pipeline, but the target is `target_delta` (next year's change). The model can no longer look accurate by copying its `co2` input; it must explain what MOVES emissions. Predictions are converted back to the level before scoring, so the comparison stays fair. Your pre-registration question 2 is about to be tested.

In [ ]:
ridge_delta_by_alpha = fit_ridge("target_delta")
for a, p in ridge_delta_by_alpha.items():
    print(f"alpha={a:>6}: val MAE {evaluate.mae(val['target_co2_next'], p):.3f}")

In [ ]:
RIDGE_DELTA_ALPHA = ...   # <- fill in
val["pred_ridge_delta"] = ridge_delta_by_alpha[RIDGE_DELTA_ALPHA]

## 5. Gradient-boosted trees

`HistGradientBoostingRegressor` with modest, defensible settings; no imputer or scaler needed (native NaN handling; splits are scale-free). We fit both parameterizations. Note what we are NOT doing: no wide hyperparameter search. The project question is whether the model generalizes through time, not whether 400 configurations can flatter one validation window; that discipline is also why Session 5's XGBoost tuning will stay small.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

def fit_hgb(target_col):
    model = HistGradientBoostingRegressor(
        max_depth=None, max_iter=300, learning_rate=0.05,
        early_stopping=False, random_state=SEED,
    )
    model.fit(train[FEATURES], train[target_col])
    pred = model.predict(val[FEATURES])
    return pred if target_col == "target_co2_next" else val["co2"].to_numpy() + pred

val["pred_hgb_level"] = fit_hgb("target_co2_next")
val["pred_hgb_delta"] = fit_hgb("target_delta")

## 6. The six-way table

Baselines recomputed in-notebook (cheap, and it guarantees the same-rows rule covers all six models simultaneously). This table is the session's artifact and overwrites `results/model_comparison.csv` in full, per the rerun convention.

In [ ]:
val["pred_persistence"] = val["co2"]
val["pred_trend"] = val["co2"] + val["co2_slope5"]

table = evaluate.comparison_table(
    val,
    {
        "persistence": "pred_persistence",
        "linear_trend": "pred_trend",
        "ridge_level": "pred_ridge_level",
        "ridge_delta": "pred_ridge_delta",
        "hgb_level": "pred_hgb_level",
        "hgb_delta": "pred_hgb_delta",
    },
    persistence_col="pred_persistence",
)
table.insert(1, "split", "val")
table.sort_values("MAE").round(3)

In [ ]:
table.to_csv("../results/model_comparison.csv", index=False)
print("saved -> results/model_comparison.csv (six models, val split)")

## 7. Reconcile and interpret

Answer in this cell, in your own words:

1. Score your four pre-registrations. What did the biggest miss misunderstand?
   > *...*
2. Level versus delta: what does the table say, and WHY? Connect it to what Ridge's co2 coefficient does in the level parameterization.
   > *...*
3. Look at MedianAE and MdAPE, not just MAE: does the model ranking change? What does that imply about who the ML models help, giants or typical countries?
   > *...*
4. The skill column: write the one honest sentence the README should carry about whether ML beats doing nothing, as of this session.
   > *...*

## Check questions (close Session 4 by answering these to Claude)

1. Why does standardizing features change Ridge's predictions but not the tree's? Where exactly in each algorithm does the difference live?
2. Bagging and boosting both combine many trees. State, in one sentence each, HOW each one reduces error, and which of the two our HistGB model is.
3. The imputer in the Ridge pipeline: on which rows was it fitted, which rows did it transform, and what specific harm occurs if it is fitted on all rows before splitting?
4. Your Ridge-on-level model almost certainly assigned a coefficient near 1 to `co2` after unscaling. What does that tell you about what it learned, and how does the delta parameterization change the question the model must answer?

**End-of-session protocol:** commit the notebook and results file; update CLAUDE.md status, decisions (chosen alphas), and session log; record the exact next action (Session 5: XGBoost with small tuning on validation, then the single test-set evaluation of the frozen winner).